In [1]:
import numpy as np
import os
import pandas as pd
from scipy.optimize import curve_fit
from LPPL_codex import LPPLModel


In [2]:
frequency= 'weekly'
in_dir = os.path.join('../data/', frequency)
ticker = '/GC'
price_col = 'close'

in_path = os.path.join(in_dir, f'{ticker.replace("/","_")}.parquet')
df = pd.read_parquet(in_path)

In [ ]:

# Prepare data
series = df.set_index('datetime')[price_col].sort_index()

# Initialize model with faster settings
model = LPPLModel(max_nfev=1000)

results = []
n = len(series)

# Loop window_end from last date back to first date + 90
for end_pos in range(n - 1, 89, -1):
    end_date = series.index[end_pos]

    # For each window length between 10 and 120
    for window_size in range(10, 121):
        start_pos = end_pos - window_size + 1
        if start_pos < 0:
            break

        window = series.iloc[start_pos:end_pos + 1]

        fit_result = model.fit(
            prices=window.values,
            n_starts=5,
            use_log=True,
            enforce_filters=True,
        )

        # progress print
        print(f"window_end={end_date}, window_size={window_size}, success={fit_result.success}")

        tc_index_ahead = np.nan if np.isnan(fit_result.tc) else fit_result.tc - (window_size - 1)
        tc_date = (end_date + pd.Timedelta(days=int(round(tc_index_ahead)))) if fit_result.success and np.isfinite(tc_index_ahead) else pd.NaT

        results.append({
            'window_end': end_date,
            'window_start': window.index[0],
            'window_size': window_size,
            'success': fit_result.success,
            'tc': fit_result.tc,
            'tc_index_ahead': tc_index_ahead,
            'tc_date': tc_date,
            'm': fit_result.m,
            'omega': fit_result.omega,
            'A': fit_result.A,
            'B': fit_result.B,
            'C1': fit_result.C1,
            'C2': fit_result.C2,
            'C': fit_result.C,
            'phi': fit_result.phi,
            'rmse': fit_result.rmse,
            'sse': fit_result.sse,
            'r2': fit_result.r2,
            'n_obs': fit_result.n_obs,
            'message': fit_result.message,
        })

# Create results dataframe
df_result = pd.DataFrame(results)


out_dir = os.path.join('../result', frequency)

# Save to parquet
output_path = os.path.join(out_dir, f'{ticker.replace("/","_")}_{frequency}_{price_col}.parquet')
df_result.to_parquet(output_path, engine='pyarrow')

print(f"Results saved to {output_path}")
print(f"Total records: {len(df_result)}")
print(df_result.head())

window_end=2026-04-06 05:00:00+00:00, window_size=10, success=False
window_end=2026-04-06 05:00:00+00:00, window_size=11, success=False
window_end=2026-04-06 05:00:00+00:00, window_size=12, success=False
window_end=2026-04-06 05:00:00+00:00, window_size=13, success=False
window_end=2026-04-06 05:00:00+00:00, window_size=14, success=False
window_end=2026-04-06 05:00:00+00:00, window_size=15, success=True
window_end=2026-04-06 05:00:00+00:00, window_size=16, success=True
window_end=2026-04-06 05:00:00+00:00, window_size=17, success=True
window_end=2026-04-06 05:00:00+00:00, window_size=18, success=True
window_end=2026-04-06 05:00:00+00:00, window_size=19, success=True
window_end=2026-04-06 05:00:00+00:00, window_size=20, success=True
window_end=2026-04-06 05:00:00+00:00, window_size=21, success=True
window_end=2026-04-06 05:00:00+00:00, window_size=22, success=True
window_end=2026-04-06 05:00:00+00:00, window_size=23, success=True
window_end=2026-04-06 05:00:00+00:00, window_size=24, suc